 # Policy Comparison

This notebook compares the performance of several trained policies on different Adroit tasks.  
It runs 100 rollouts per policy and visualizes the results using boxplots.

## Import libraries

Import libraries for dataset handling, policy loading, and plotting.

In [ ]:
import minari
import d3rlpy
import numpy as np
import matplotlib.pyplot as plt
import os

## Experiment setup

Define the type of experiment, selected tasks, and algorithms to compare

In [ ]:
experiment = 'offline'  # 'offline', 'finetuning', or 'online'

tasks = ['relocate', 'door', 'pen', 'hammer']
algorithms = ['iql', 'cql', 'td3bc', 'awac', 'bc']

In [ ]:
# Initialize dictionaries to store loaded policies and environments
policies = {}
environments = {}

## Rollout parameters

Define the number of episodes per policy to evaluate

In [ ]:
# Number of episodes during the testing phase
N = 100

## Output folder

Create a folder to store rollout results.

In [ ]:
# Define the folder where rollout results will be saved
path = os.path.join("rollout", experiment)

# Create the folder if it doesn't already exist
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Created: {path}")
else:
    print(f"Already exists: {path}")

## Load Policies and Environments

Load the saved policies and corresponding environments

In [ ]:
for task in tasks:
    # Initialize policy dictionary for the current task
    policies[task] = {}

    # Recover the environment for the task using the human-v2 dataset
    environments[task] = minari.load_dataset(f"D4RL/{task}/human-v2").recover_environment()

    for algorithm in algorithms:
        # Load the corresponding trained policy for each algorithm
        policies[task][algorithm] = d3rlpy.load_learnable(f"policies/{experiment}/{task}_{algorithm}_policy.d3")

## Rollout function

Evaluate a policy over N episodes with fixed random seeds

In [ ]:
def evaluate_policies(pol, env):

    all_rewards = []

    # Generate a list of random seeds to make the episodes reproducible
    seeds = np.random.randint(0, 10000, size=N)

    # Loop over episodes
    for episode_idx in range(N):
        seed = seeds[episode_idx]
        
        # Evaluate each policy on the same episode (same seed)
        obs, _ = env.reset(seed=int(seed))
        done = False
        total_reward = 0.0
        step_count = 0

        # Perform rollout until the end of the episode
        while not done:
            action = pol.predict(obs[None])[0]
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            step_count += 1
            done = terminated or truncated

        # Compute and store the average reward per step
        avg_reward = total_reward / step_count if step_count > 0 else 0.0
        all_rewards.append(total_reward)

    # Return the collected average rewards for each policy
    return all_rewards

## Evaluation loop

Run rollouts for each task and algorithm, storing rewards

In [ ]:
# Dictionary to store evaluation results
rewards = {}

for task in tasks:
    rewards[task] = {}
    for algorithm in algorithms:
        # Evaluate each policy in its respective environment
        rewards[task][algorithm] = evaluate_policies(policies[task][algorithm], environments[task])

## Plot colors

Define color scheme for each algorithm in the boxplot.

In [ ]:
# Assign a specific color to each algorithm for consistent plotting
colors = {
    'iql': 'tab:blue',
    'cql': 'tab:purple',
    'bc': 'tab:green',
    'td3bc': 'tab:red',
    'awac': 'tab:orange'
}

## Boxplot generation

Generate one boxplot per task, illustrating the average total reward per episode for each algorithm.

In [ ]:
# Generate a boxplot for each task
for task in tasks:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Prepare data: list of reward lists, one per algorithm
    task_data = [rewards[task][algo] for algo in algorithms]

    # Print summary statistics for each algorithm
    print(f"\nTask: {task}")
    for algo, data in zip(algorithms, task_data):
        median = np.median(data)
        std = np.std(data)
        print(f"{algo}: median = {median:.2f}, std = {std:.2f}")
    
    # Create boxplot
    box = ax.boxplot(task_data, patch_artist=True, tick_labels=algorithms)
    
    # Color each box according to the algorithm
    for patch, median_line, flier, algo in zip(box['boxes'], box['medians'], box['fliers'], algorithms):
        color = colors.get(algo, 'gray')

        patch.set_facecolor(color)
        patch.set_alpha(0.7)

        median_line.set_color(color)
        median_line.set_linewidth(2.5)

        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor('black')
        flier.set_alpha(0.8)
        flier.set_markersize(6)

    ax.set_title(f'Average reward per episode — {experiment} — {task}')
    ax.set_ylabel('Average Reward')
    ax.grid(True, axis='y')

    # Dynamically set Y-axis limits for better visibility
    all_values = [val for sublist in task_data for val in sublist]
    ax.set_ylim(min(all_values) * 1.2 if min(all_values) < 0 else 0, max(all_values) * 1.2)

    # Save plot as PNG
    plt.savefig(f'rollout/{experiment}/{task}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)